In [1]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [12]:
!pip install -e .

Obtaining file:///content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/SliceGPTModifications
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached transformers-4.41.0-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.41.0-py3-none-any.whl (9.1 MB)
Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
  Building editable for transformercompression (pyproject.toml) ... done
  Created wheel for transformercompression: filename=transformercompression-0.0.1-0.editable-py3-none-any.whl size=5802 sha256=01ef1eb15e2d1ce12fa8de721b705537735d5cd2320b669faac60fb6fe815067
  Stored in directory: /tmp/pip-ephem-wheel-cache-k1npaqbz/wheels/fa/

In [2]:
import slicegpt
from slicegpt import rotate, model_utils

In [3]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "gemma3_slicing", "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "gemma3_slicing", "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [4]:
!pip install "transformers>=4.50.0"

  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.41.0
    Uninstalling transformers-4.41.0:
      Successfully uninstalled transformers-4.41.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformercompression 0.0.1 requires transformers==4.41.0, but you have transformers 4.57.3 which is incompatible.


In [4]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get("HF_TOKEN"))

In [5]:
import os
import subprocess
from datetime import datetime

MODEL_ID = "google/gemma-3-270m"

sparsities = [0.10]
datasets = ["squad"]

def run_slicegpt(dataset, sparsity):
    log_name = f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}".replace(".", "p") + ".txt"
    log_path = os.path.join(LOG_DIR, log_name)

    save_dir = os.path.join(MODEL_DIR, f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}".replace(".", "p"))
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python",
        "/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", MODEL_ID,
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--cal-batch-size", str(8)
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            f.write(line)

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)


Running: python /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py --model google/gemma-3-270m --cal-dataset squad --save-dir /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/gemma3_slicing/models/google-gemma-3-270m_squad_s0p10 --sparsity 0.1 --device cuda:0 --no-wandb --cal-batch-size 8
Log file: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/gemma3_slicing/logs/google-gemma-3-270m_squad_s0p10.txt
Start: 2026-01-10 20:12:17.311600

2026-01-10 20:12:24.902886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768075944.926484   24679 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768075944.933471   24679 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory fo

In [6]:
import torch
from slicegpt import hf_utils

model_path = os.path.join(MODEL_DIR, "google-gemma-3-270m_squad_s0p10")

model_adapter, tokenizer = hf_utils.load_sliced_model(
    "google/gemma-3-270m",
    model_path,
    sparsity=0.10
)

model = model_adapter.model.to(device="cuda", dtype=torch.float16).eval()

`torch_dtype` is deprecated! Use `dtype` instead!


In [12]:
prompt = "What is "
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(gen[0], skip_special_tokens=True)

print(f"Input:\n{prompt}")
print(f"Output:\n{output}")

Input:
What is 
Output:
What is 
